# Preprocessing for Clustering

This notebook creates the first model-ready preprocessing output from the customer-level feature table. It does not run clustering or modelling.

## Imports

Use the reusable preprocessing functions from `src/preprocessing.py`.

In [1]:
import sys

import pandas as pd

sys.path.append("../src")

from preprocessing import FINAL_MODEL_COLUMNS, clean_feature_values, preprocess_for_clustering, select_model_features


## Load Customer Features

Load the combined customer-level feature table created during feature engineering.

In [2]:
customer_features = pd.read_csv("../data/processed/customer_features_info.csv")
customer_features.head()


,customer_id,customer_gender,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_total_distinct_products,year_first_transaction,percentage_of_products_bought_promotion,typical_hour,...,share_electronics,share_vegetables,share_nonalcohol_drinks,share_alcohol_drinks,share_meat,share_fish,share_hygiene,share_videogames,share_petfood,degree_level
0,3,female,1.0,1.0,1.0,3.0,189.0,2020.0,0.631599,NaN,...,0.244917,0.020065,0.017375,0.009521,0.001506,0.011458,0.029693,0.013771,0.020656,Bsc
1,4,female,1.0,0.0,0.0,2.0,130.0,2013.0,0.149890,NaN,...,0.047596,0.099442,0.026343,0.004695,0.002125,0.000741,0.092918,0.016458,0.032867,Bsc
2,5,male,0.0,0.0,NaN,2.0,81.0,2005.0,0.069126,11.0,...,0.000000,0.035694,0.006496,0.007589,0.081356,0.017557,0.032607,0.006496,0.014277,Msc
3,7,male,0.0,0.0,2.0,1.0,92.0,2021.0,0.253609,18.0,...,0.073903,0.005618,0.050629,0.075776,0.065008,0.072432,0.032437,0.110754,0.012306,Unknown
4,8,male,0.0,0.0,3.0,1.0,6.0,2021.0,0.186569,17.0,...,0.420243,0.014730,0.022948,0.027833,0.041400,0.039346,0.011513,0.048765,0.017095,Unknown


## Initial Checks

Check shape, duplicate customer IDs, and missing values before preprocessing.

In [3]:
print(f"Rows before preprocessing: {customer_features.shape[0]:,}")
print(f"Columns before preprocessing: {customer_features.shape[1]:,}")
print(f"Duplicated customer_id before preprocessing: {customer_features['customer_id'].duplicated().sum():,}")

Rows before preprocessing: 33,038
Columns before preprocessing: 29
Duplicated customer_id before preprocessing: 0


In [4]:
missing_before = customer_features.isna().sum().sort_values(ascending=False)
missing_before[missing_before > 0]

share_fish                                 991
typical_hour                               661
share_meat                                 661
share_electronics                          661
number_complaints                          661
share_vegetables                           661
share_videogames                           661
share_petfood                              661
total_children_home                        656
percentage_of_products_bought_promotion    330
teens_home                                 330
distinct_stores_visited                    330
kids_home                                  330
share_alcohol_drinks                       330
share_hygiene                              330
age                                        165
dtype: int64

## Clean and Select Features

Clean invalid and missing values, then remove non-model columns while keeping `customer_id` for traceability.

In [5]:
cleaned_customer_features = clean_feature_values(customer_features)
selected_customer_features = select_model_features(cleaned_customer_features)

removed_columns = [
    column for column in customer_features.columns
    if column not in selected_customer_features.columns
]

removed_columns


['customer_gender',
 'kids_home',
 'teens_home',
 'year_first_transaction',
 'latitude',
 'longitude',
 'has_children',
 'degree_level']

In [6]:
missing_after_cleaning = selected_customer_features.isna().sum().sort_values(ascending=False)
missing_after_cleaning[missing_after_cleaning > 0]

Series([], dtype: int64)

## Preprocess Model Features

Create the final compact model input. Gender and degree fields are excluded from clustering, and the 20 numeric business features are scaled with `RobustScaler`. `customer_id` is preserved only for traceability.


In [7]:
selected_model_features = preprocess_for_clustering(customer_features)

expected_columns = ["customer_id", *FINAL_MODEL_COLUMNS]
if selected_model_features.columns.tolist() != expected_columns:
    raise ValueError("selected_model_features.csv does not match the final compact feature specification.")

selected_model_features.head()


,customer_id,number_complaints,distinct_stores_visited,lifetime_total_distinct_products,percentage_of_products_bought_promotion,typical_hour,age,customer_tenure,has_loyalty_card,total_children_home,...,share_groceries,share_electronics,share_vegetables,share_nonalcohol_drinks,share_alcohol_drinks,share_meat,share_fish,share_hygiene,share_videogames,share_petfood
0,3,0.0,0.0,0.461538,1.147251,0.000,0.03125,-0.714286,0.0,0.0,...,-0.350008,1.556431,-0.034449,-0.096317,-0.432541,-0.956394,-0.432012,-0.038953,0.304556,0.392400
1,4,-1.0,-0.5,0.048951,-0.262009,0.000,-0.12500,0.285714,0.0,-1.0,...,-0.130392,-0.254970,1.700878,0.440933,-0.591465,-0.936412,-0.815459,1.696860,0.530206,1.256560
2,5,0.0,-0.5,-0.293706,-0.498290,-0.125,0.00000,1.428571,-1.0,-2.0,...,0.450655,-0.691895,0.307231,-0.748057,-0.496175,1.621038,-0.213760,0.041029,-0.306310,-0.059029
3,7,1.0,-1.0,-0.216783,0.041426,0.750,-0.34375,-0.857143,0.0,-2.0,...,-0.973212,-0.013467,-0.350277,1.895790,1.749301,1.093357,1.749713,0.036376,8.447787,-0.198545
4,8,2.0,-1.0,-0.818182,-0.154704,0.625,0.06250,-0.857143,0.0,-2.0,...,-1.668903,3.165917,-0.151063,0.237569,0.170477,0.331328,0.565851,-0.538095,3.242873,0.140370


## Final Validation

Confirm the preprocessed output keeps all customers, has no duplicate IDs, and has no missing values in model features.

In [8]:
preprocessing_validation = pd.DataFrame(
    {
        "check": [
            "rows before preprocessing",
            "rows after preprocessing",
            "columns before preprocessing",
            "selected columns including customer_id",
            "duplicated customer_id after preprocessing",
            "missing values in selected model features",
        ],
        "value": [
            customer_features.shape[0],
            selected_model_features.shape[0],
            customer_features.shape[1],
            selected_model_features.shape[1],
            selected_model_features["customer_id"].duplicated().sum(),
            selected_model_features.drop(columns=["customer_id"]).isna().sum().sum(),
        ],
    }
)

preprocessing_validation


,check,value
0,rows before preprocessing,33038
1,rows after preprocessing,33038
2,columns before preprocessing,29
3,selected columns including customer_id,21
4,duplicated customer_id after preprocessing,0
5,missing values in selected model features,0


In [9]:
missing_after = selected_model_features.drop(columns=["customer_id"]).isna().sum().sort_values(ascending=False)
missing_after[missing_after > 0]


Series([], dtype: int64)

In [10]:
selected_model_features.columns.to_frame(index=False, name="feature_column")


,feature_column
0,customer_id
1,number_complaints
2,distinct_stores_visited
3,lifetime_total_distinct_products
4,percentage_of_products_bought_promotion
5,typical_hour
6,age
7,customer_tenure
8,has_loyalty_card
9,total_children_home


## Save Preprocessed Feature Table

Save the preprocessed customer-level feature table. This file keeps `customer_id` for traceability, but clustering should use the remaining feature columns.

In [11]:
selected_model_features.to_csv("../data/processed/selected_model_features.csv", index=False)
print("Saved ../data/processed/selected_model_features.csv")


Saved ../data/processed/selected_model_features.csv


## Preprocessing Notes

- `customer_id` is kept in the final file for traceability, but it is excluded from model feature scaling.
- Basket-derived features are excluded from clustering and kept for post-clustering profiling and promotions.
- Gender and degree fields are excluded from the final clustering input after feature sensitivity checks.
- Numeric missing values are filled with the median, and categorical missing values are filled with `Unknown` before selection.
- Invalid `percentage_of_products_bought_promotion` values are clipped to the `0` to `100` range.
- The 20 final numeric business features are scaled with `RobustScaler`.
- No clustering or modelling is performed in this notebook.
